# Vehicle Detection model using yolov11

get ultralytics (yolov11 import)

In [ ]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.6/922.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

Install the requirements

In [ ]:
import ultralytics
ultralytics.checks()


Ultralytics 8.3.86 🚀 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 39.1/112.6 GB disk)


Load the datasets

In [ ]:
# prompt: connect to google drive

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


Splitting the datasets

In [ ]:
import os
import shutil

# Define the base dataset path (adjust as needed)
base_dir = "/content/drive/MyDrive/vehicle-detection.v3i.yolov11"

# Define the images and labels directories
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

# List of subdirectories to combine (e.g., "train" and "val")
subdirs = ["train", "val"]

def combine_subdirs(parent_dir, subdirs):
    for sub in subdirs:
        sub_dir_path = os.path.join(parent_dir, sub)
        if not os.path.isdir(sub_dir_path):
            print(f"Subdirectory not found: {sub_dir_path}")
            continue
        # Move each file from the subdirectory to the parent directory
        for file in os.listdir(sub_dir_path):
            src_file = os.path.join(sub_dir_path, file)
            dst_file = os.path.join(parent_dir, file)
            # If file with same name exists, you can choose to overwrite or skip.
            if os.path.exists(dst_file):
                print(f"File {dst_file} already exists. Skipping {src_file}.")
            else:
                shutil.move(src_file, dst_file)
        # Remove the now empty subdirectory
        os.rmdir(sub_dir_path)
        print(f"Removed directory: {sub_dir_path}")

# Combine images and labels back into the parent directories
combine_subdirs(images_dir, subdirs)
combine_subdirs(labels_dir, subdirs)

print("Files from train and val subdirectories have been combined into the parent directory.")


Removed directory: /content/drive/MyDrive/vehicle-detection.v3i.yolov11/images/train
Removed directory: /content/drive/MyDrive/vehicle-detection.v3i.yolov11/images/val
Removed directory: /content/drive/MyDrive/vehicle-detection.v3i.yolov11/labels/train
Removed directory: /content/drive/MyDrive/vehicle-detection.v3i.yolov11/labels/val
Files from train and val subdirectories have been combined into the parent directory.


In [ ]:
import os
import random
import shutil

random.seed(42)

dataset_path = "/content/drive/MyDrive/vehicle-detection.v3i.yolov11"

images_dir = os.path.join(dataset_path, "images")
labels_dir = os.path.join(dataset_path, "labels")

splits = ["train", "val", "test"]

for split in splits:
    os.makedirs(os.path.join(images_dir, split), exist_ok=True)
    os.makedirs(os.path.join(labels_dir, split), exist_ok=True)

image_files = [f for f in os.listdir(images_dir)
               if f.lower().endswith((".jpg", ".png")) and os.path.isfile(os.path.join(images_dir, f))]

random.shuffle(image_files)
num_images = len(image_files)

train_ratio = 0.7
val_ratio   = 0.15
# Test ratio will be the remaining images

num_train = int(num_images * train_ratio)
num_val   = int(num_images * val_ratio)
num_test  = num_images - num_train - num_val

train_files = image_files[:num_train]
val_files   = image_files[num_train:num_train + num_val]
test_files  = image_files[num_train + num_val:]

def move_files(file_list, split_name):
    for file in file_list:
        src_image = os.path.join(images_dir, file)
        dst_image = os.path.join(images_dir, split_name, file)
        shutil.move(src_image, dst_image)

        label_file = os.path.splitext(file)[0] + ".txt"
        src_label = os.path.join(labels_dir, label_file)
        if os.path.exists(src_label):
            dst_label = os.path.join(labels_dir, split_name, label_file)
            shutil.move(src_label, dst_label)

move_files(train_files, "train")
move_files(val_files, "val")
move_files(test_files, "test")

print("Dataset successfully split:")
print(f"Train: {len(train_files)} images")
print(f"Validation: {len(val_files)} images")
print(f"Test: {len(test_files)} images")


Dataset successfully split:
Train: 6450 images
Validation: 1382 images
Test: 1383 images


In [ ]:
import os
import shutil

# Define the base dataset directory (adjust as needed)
base_dir = "/content/drive/MyDrive/vehicle-detection.v3i.yolov11"

# List of splits to process
splits = ["train", "val", "test"]

for split in splits:
    # Create new directories: train/images and train/labels, etc.
    new_split_dir = os.path.join(base_dir, split)
    new_images_dir = os.path.join(new_split_dir, "images")
    new_labels_dir = os.path.join(new_split_dir, "labels")
    os.makedirs(new_images_dir, exist_ok=True)
    os.makedirs(new_labels_dir, exist_ok=True)

    # Define old directories for images and labels for this split
    old_images_dir = os.path.join(base_dir, "images", split)
    old_labels_dir = os.path.join(base_dir, "labels", split)

    # Move image files from the old images directory to the new one
    if os.path.exists(old_images_dir):
        for file in os.listdir(old_images_dir):
            src_file = os.path.join(old_images_dir, file)
            dst_file = os.path.join(new_images_dir, file)
            shutil.move(src_file, dst_file)
        # Remove the old images subdirectory
        os.rmdir(old_images_dir)

    # Move label files from the old labels directory to the new one
    if os.path.exists(old_labels_dir):
        for file in os.listdir(old_labels_dir):
            src_file = os.path.join(old_labels_dir, file)
            dst_file = os.path.join(new_labels_dir, file)
            shutil.move(src_file, dst_file)
        # Remove the old labels subdirectory
        os.rmdir(old_labels_dir)

# Optionally, remove the now empty base 'images' and 'labels' directories if desired.
old_images_parent = os.path.join(base_dir, "images")
old_labels_parent = os.path.join(base_dir, "labels")
if os.path.exists(old_images_parent) and not os.listdir(old_images_parent):
    os.rmdir(old_images_parent)
if os.path.exists(old_labels_parent) and not os.listdir(old_labels_parent):
    os.rmdir(old_labels_parent)

print("Dataset structure updated successfully!")


Dataset structure updated successfully!


Training the model - YOLOV11 🚀

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt")

100%|██████████| 5.35M/5.35M [00:00<00:00, 66.5MB/s]


In [ ]:
results = model.train(data="/content/drive/MyDrive/vehicle-detection.v3i.yolov11/data.yaml", epochs=30, imgsz=640, batch = 8)

Ultralytics 8.3.86 🚀 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/drive/MyDrive/vehicle-detection.v3i.yolov11/data.yaml, epochs=30, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=Fal

100%|██████████| 755k/755k [00:00<00:00, 14.7MB/s]


Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

train: Scanning /content/drive/MyDrive/vehicle-detection.v3i.yolov11/train/labels... 6448 images, 3 backgrounds, 0 corrupt: 100%|██████████| 6450/6450 [05:02<00:00, 21.29it/s]


train: New cache created: /content/drive/MyDrive/vehicle-detection.v3i.yolov11/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

      30/30      1.31G      1.503      0.935      1.267         16        640: 100%|██████████| 807/807 [01:19<00:00, 10.13it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 87/87 [00:07<00:00, 11.00it/s]

                   all       1382       8983       0.69      0.713      0.751      0.439



30 epochs completed in 0.743 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train/weights/best.pt, 5.5MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.86 🚀 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 87/87 [00:09<00:00,  9.61it/s]


                   all       1382       8983      0.689      0.712      0.751       0.44
                   bus        699       1018      0.789      0.883        0.9      0.608
                   car       1009       4237      0.616      0.799      0.749       0.39
              microbus        423        549       0.67      0.664      0.721      0.449
             motorbike        845       2292      0.681      0.597      0.695      0.295
            pickup-van        491        660      0.619       0.63      0.662      0.379
                 truck        176        227      0.759        0.7      0.778      0.518
Speed: 0.1ms preprocess, 0.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/detect/train


Store the weights in google drive

In [ ]:
%%bash
export LC_ALL=en_US.UTF-8
export LANG=en_US.UTF-8
cp -r /content/runs /content/drive/MyDrive/

In [ ]:
%%bash
export LC_ALL=en_US.UTF-8
export LANG=en_US.UTF-8
cp /content/runs/detect/train/weights/best.pt /content/drive/MyDrive/vehicle_detect.pt
